In [43]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from .env
load_dotenv()

# Verify the key is loading (shows only the first 5 chars)
key = os.getenv("OPENAI_API_KEY")
print(f"Key loaded: {key[:5]}...") 
print(f"Base URL: {os.getenv('OPENAI_BASE_URL')}")

client = OpenAI(
    api_key=key,
    base_url=os.getenv("OPENAI_BASE_URL")
)

t0 = time.perf_counter()
try:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user', 'content': 'Reply with the single word: ok'}]
    )
    print("Response:", r.choices[0].message.content)
    print("Latency:", round(time.perf_counter() - t0, 2), "s")
except Exception as e:
    print("Error:", e)

Key loaded: sk-SD...
Base URL: https://api.ohmygpt.com/v1
Response: ok
Latency: 1.4 s


In [44]:
import pymupdf
import re
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent
PDF = ROOT / 'data' / 'raw' / 'osh_act_2006.pdf'

doc = pymupdf.open(PDF)
print('Total pages:', len(doc))
pages = [{'page': i, 'text': p.get_text('text')} for i, p in enumerate(doc, start=1)]

counts = Counter()
for p in pages:
    for ln in {l.strip() for l in p['text'].splitlines() if l.strip()}:
        counts[ln] += 1

# FIX: Exclude sub-numbers like (1), (a) from being classified as headers/footers
repeated = {ln for ln, c in counts.items() if c > 0.3 * len(pages) and not re.match(r'^\([a-zA-Z0-9]+\)$', ln.strip())}

print(f'\nFound {len(repeated)} true headers/footers:')
print(sorted(repeated))

for p in pages:
    p['clean'] = '\n'.join(
        l for l in p['text'].splitlines()
        if l.strip() and not re.fullmatch(r'\d{1,3}', l.strip()) and l.strip() not in repeated
    )

out_path = ROOT / 'data' / 'processed' / 'pages.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    for p in pages:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f'\nSaved {len(pages)} cleaned pages to: {out_path}')

Total pages: 50

Found 3 true headers/footers:
['By Ulii.org and Laws.Africa. Share widely and freely.', 'Occupational Safety and Health Act (Chapter 231)', 'Uganda']

Saved 50 cleaned pages to: c:\Users\Kagimu Brian\osh_rag_project\data\processed\pages.jsonl


In [45]:
from collections import Counter

# Regex for Parts
PART_RE = re.compile(r'^PART\s+([IVXLC]+)\s*[-–—]?\s*(.*)$', re.I)

full = '\n'.join(p['clean'] for p in pages)
start = full.upper().find('BE IT ENACTED')

if start == -1:
    print("ERROR: Could not find 'BE IT ENACTED'.")
else:
    lines = full[start:].splitlines()
    
    chunks, cur = [], None
    part_no = part_title = ''
    last_sec = 0
    skip_next = False

    def flush():
        global cur
        if cur and cur['lines']:
            chunks.append({
                'doc': 'OSH Act 2006',
                'part_no': cur['part_no'],
                'part_title': cur['part_title'],
                'section': cur['section'],
                'section_title': cur['title'],
                'text': '\n'.join(cur['lines']).strip()
            })
        cur = None

    for i, ln in enumerate(lines):
        if skip_next:
            skip_next = False
            continue

        s = ln.strip()
        
        # FIX: Only break if it matches "SCHEDULE 1" or "SCHEDULE 2", etc., AND we are past section 121
        if re.match(r'^SCHEDULE\s+[1-9]', s, re.I) and last_sec >= 121:
            break

        m = PART_RE.match(s)
        if m:
            part_no, part_title = m.group(1).upper(), m.group(2).strip()
            continue
            
        # Pattern 1: Number on its own line
        m_sec_num = re.match(r'^(\d{1,3})\s*\.$', s)
        if m_sec_num and int(m_sec_num.group(1)) == last_sec + 1:
            flush()
            last_sec += 1
            title = ""
            if i + 1 < len(lines):
                title = lines[i+1].strip()
                skip_next = True
            
            cur = {'section': last_sec, 'title': title, 'part_no': part_no, 'part_title': part_title, 'lines': [s, title]}
            continue

        # Pattern 2: Number and title on the same line
        m_sec_inline = re.match(r'^(\d{1,3})\s*\.\s+(.+)$', s)
        if m_sec_inline and int(m_sec_inline.group(1)) == last_sec + 1:
            flush()
            last_sec += 1
            cur = {'section': last_sec, 'title': m_sec_inline.group(2).strip(), 'part_no': part_no, 'part_title': part_title, 'lines': [s]}
            continue

        if cur:
            cur['lines'].append(s)

    flush()
    
    print('Sections found:', len(chunks))
    missing = sorted(set(range(1, 122)) - {c['section'] for c in chunks})
    print('Missing:', missing)

Sections found: 121
Missing: []


In [46]:
# Find all occurrences of '121' in the text and show the surrounding lines
import re

lines = full.splitlines()
for i, ln in enumerate(lines):
    if '121' in ln:
        # Print the line before, the line itself, and the line after
        print(f"--- Line {i} ---")
        if i > 0: print("PREV:", repr(lines[i-1]))
        print("CURR:", repr(lines[i]))
        if i < len(lines)-1: print("NEXT:", repr(lines[i+1]))

--- Line 158 ---
PREV: '120. Minister’s power to amend schedules ...................................................................................................................................... 37'
CURR: '121. Repeal of Cap 220 and savings .................................................................................................................................................. 38'
NEXT: 'Schedule ................................................................................................................................................................................................................ 38'
--- Line 2239 ---
PREV: 'Schedules 2, 3, 4, 5, 6, 7, and 8.'
CURR: '121.'
NEXT: 'Repeal of Cap 220 and savings'


In [47]:
start = full.upper().find('BE IT ENACTED')

if start != -1:
    lines = full[start:].splitlines()
    print("--- RAW TEXT SAMPLE ---")
    # Print lines 10 to 30 to see how sections are formatted
    for i in range(10, 35):
        if i < len(lines):
            print(repr(lines[i]))
else:
    print("Could not find 'BE IT ENACTED'")

--- RAW TEXT SAMPLE ---
'for containing compressed air, connected with an air compressing plant or any ﬁxed vessel for containing'
'compressed air or compressed exhaust gases, used for the purpose of starting an internal combustion'
'engine;'
'“area under the inﬂuence of” includes any area where the fall of the contaminant released directly or'
'indirectly from an undertaking may come to rest or be present and cause its deleterious effect whether'
'in its original form or in a chemically modiﬁed form through natural processes having been carried there'
'by prevailing wind, rain water run-off or by any other natural agent, and any area where dangerous levels'
'of the contaminant may be present and cause its effects, having been taken there through the efﬂuent or'
'other waste disposal methods used at the undertaking;'
'“article” means solid, liquid or gas or any of their combination or any plant designed for use or operation'
'(whether exclusively or not) by persons at work or any artic

In [48]:
start = full.upper().find('BE IT ENACTED')

if start != -1:
    lines = full[start:].splitlines()
    print("--- FIRST 15 LINES ---")
    for i in range(0, 15):
        if i < len(lines):
            print(f"Line {i}: {repr(lines[i])}")

--- FIRST 15 LINES ---
Line 0: 'D by Parliament as follows:'
Line 1: 'Part I – Preliminary'
Line 2: '1.'
Line 3: 'Commencement'
Line 4: 'This Act shall come into force on a date to be appointed by the Minister by statutory instrument and'
Line 5: 'different dates may be appointed for the commencement of different provisions.'
Line 6: '2.'
Line 7: 'Interpretation'
Line 8: 'In this Act, unless the context otherwise requires—'
Line 9: '“air receiver” means any vessel (other than a pipe or coil, or an accessory, ﬁtting or part of a compressor)'
Line 10: 'for containing compressed air, connected with an air compressing plant or any ﬁxed vessel for containing'
Line 11: 'compressed air or compressed exhaust gases, used for the purpose of starting an internal combustion'
Line 12: 'engine;'
Line 13: '“area under the inﬂuence of” includes any area where the fall of the contaminant released directly or'
Line 14: 'indirectly from an undertaking may come to rest or be present and cause its deleteri

In [49]:
import json

# Rough token estimator
def n_tokens(text):
    return len(text.split())

# Regex to find subsections like (1), (2), (a)
SUB_RE = re.compile(r'^\s*\(([a-zA-Z0-9]+)\)')

def split_section(c):
    # If section is already short, don't split it
    if n_tokens(c['text']) <= 500:
        return [dict(c, subsection='')]
    
    blocks, sub, buf = [], '', []
    for ln in c['text'].splitlines():
        m = SUB_RE.match(ln)
        if m and buf:
            blocks.append((sub, '\n'.join(buf)))
            buf = []
        if m:
            sub = m.group(1)
        buf.append(ln)
    if buf:
        blocks.append((sub, '\n'.join(buf)))

    groups, g = [], []
    for b in blocks:
        if g and n_tokens('\n'.join(x[1] for x in g + [b])) > 400:
            groups.append(g)
            g = []
        g.append(b)
    if g:
        groups.append(g)

    out = []
    for g in groups:
        subs = [s for s, _ in g if s]
        label = ''
        if subs:
            if len(subs) == 1:
                label = f"({subs[0]})"
            else:
                label = f"({subs[0]})-({subs[-1]})"
        
        out.append(dict(c, text='\n'.join(t for _, t in g), subsection=label))
    return out

# The date from the PDF metadata we saw earlier
VERSION_DATE = '8 June 2006' 

final_chunks = []
for c in chunks:
    pieces = split_section(c)
    for k, d in enumerate(pieces, start=1):
        # Create a unique ID
        d['id'] = f"s{d['section']}" if len(pieces) == 1 else f"s{d['section']}_{k}"
        
        # Create the legal citation
        d['citation'] = f"OSH Act 2006, Part {d['part_no']}, s.{d['section']}{d['subsection']}"
        
        # Add a header to the text for better embedding
        d['embed_text'] = f"{d['citation']} | {d['section_title']}\n{d['text']}"
        
        d['source_file'] = 'osh_act_2006.pdf'
        d['version_date'] = VERSION_DATE
        
        final_chunks.append(d)

# Save to data/processed/chunks.jsonl
out_path = ROOT / 'data' / 'processed' / 'chunks.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for d in final_chunks:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print(f"{len(final_chunks)} chunks saved.")
print(f"Longest chunk: {max(n_tokens(d['text']) for d in final_chunks)} tokens")

129 chunks saved.
Longest chunk: 1560 tokens


In [50]:
import random
random.seed(42)

print("--- 10 RANDOM CHUNKS ---")
for d in random.sample(final_chunks, 10):
    print(f"[{d['citation']}] | {d['section_title']}")
    print(d['text'][:150].replace('\n', ' ') + "...\n")

--- 10 RANDOM CHUNKS ---
[OSH Act 2006, Part V, s.28] | General duties of manufactures, suppliers and transporters
28. General duties of manufactures, suppliers and transporters (1) It is the duty of a person who designs, manufactures, imports or supplies any artic...

[OSH Act 2006, Part II, s.6(j)-(iii)] | Powers of inspectors
(j) to take samples of any article or substance found in any premises which the inspector has power to enter, and of the atmosphere in the premises or...

[OSH Act 2006, Part XI, s.69(1)-(a)] | Hoists and lifts
69. Hoists and lifts (1) Where a hoist or lift is newly taken into use in any premises (other than premises forming part of a workplace), the occupier...

[OSH Act 2006, Part XI, s.61] | Fencing of dangerous machinery, plant, and equipment
61. Fencing of dangerous machinery, plant, and equipment (1) Every ﬂywheel directly connected to any prime mover and every moving part of any prime mo...

[OSH Act 2006, Part IX, s.56(10)-(15)] | General safety require